In [2]:
# %pip install google-play-scraper
# %pip install pandas
# %pip install tensorflow

In [3]:
from google_play_scraper import Sort, reviews, reviews_all, app
import json, pprint, time, re, nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
app_args = {
    "app_id": "com.btpn.dc",
    "lang": "en",
    "country": "id"
}

In [5]:
app_info = app(**app_args)

pd.DataFrame([app_info])
# print(json.dumps(app_info, indent=2))

,title,description,descriptionHTML,summary,installs,minInstalls,realInstalls,score,ratings,reviews,...,contentRatingDescription,adSupported,containsAds,released,lastUpdatedOn,updated,version,comments,appId,url
0,Jenius,The simple way to manage your life/finance wit...,The simple way to manage your life/finance wit...,The simple way to manage your life/finance wit...,"10,000,000+",10000000,13761625,3.27501,203279,129301,...,None,False,False,"Aug 10, 2016","Mar 13, 2026",1773408818,4.64.0,[],com.btpn.dc,https://play.google.com/store/apps/details?id=...


In [6]:
# pretty print 5 reviews based on 1* rating
result, continuation_token = reviews(
    **app_args,
    sort=Sort.RATING, # defaults to Sort.NEWEST
    count=5, # defaults to 100
    filter_score_with=1 # defaults to None(means all score)
)

result, _ = reviews(
    app_args['app_id'],
    continuation_token=continuation_token # defaults to None(load from the beginning)
)

pprint.pprint(result)

[{'appVersion': '4.63.1',
  'at': datetime.datetime(2026, 2, 23, 23, 5, 21),
  'content': 'It was good. but getting slower now. even scanning QRIS took '
             'longer compared to other mobile banking apps.',
  'repliedAt': datetime.datetime(2026, 2, 23, 23, 7, 19),
  'replyContent': 'Hi Benny Herlambang. We apologize for the inconvenience. '
                  'The information you provided has been noted and will be '
                  'used as feedback for Jenius to provide/improve better '
                  'services and facilities.',
  'reviewCreatedVersion': '4.63.1',
  'reviewId': '8a566d92-07ef-4275-9731-49bc9c34a2f3',
  'score': 1,
  'thumbsUpCount': 0,
  'userImage': 'https://play-lh.googleusercontent.com/a-/ALV-UjX8vrwAXJftZ8y96Nu6-Q1tOmHECQ9TuedRZLRByohZF_bVgpe0Ew',
  'userName': 'Benny Herlambang'},
 {'appVersion': '4.55.0',
  'at': datetime.datetime(2025, 12, 11, 20, 12, 22),
  'content': 'Nothing works, when i ask for help no one knows what to do, '
             'no

In [7]:
# collect reviews with loop safeguards
all_reviews = []
review_ids = set()

target_count = 10000
batch_size = 200
max_batches_per_sort = (target_count // batch_size) * 3
max_runtime_seconds = 20 * 60
window_size = 10
min_new_per_window = 20

start_time = time.time()

for sort_type in [Sort.NEWEST, Sort.MOST_RELEVANT]:
  continuation_token = None
  seen_tokens = set()
  new_in_window = 0
  consecutive_errors = 0

  for batch_idx in range(1, max_batches_per_sort + 1):
    if len(all_reviews) >= target_count:
      break

    if time.time() - start_time > max_runtime_seconds:
      print("Runtime limit reached. Stopping scrape early.")
      break

    try:
      result, next_token = reviews(
        **app_args,
        sort=sort_type,
        count=batch_size,
        continuation_token=continuation_token
      )
      consecutive_errors = 0
    except Exception as err:
      consecutive_errors += 1
      print(f"{sort_type.name} batch {batch_idx} failed: {err}")
      if consecutive_errors >= 3:
        print(f"{sort_type.name}: too many consecutive errors, stopping.")
        break
      time.sleep(2)
      continue

    if len(result) == 0:
      print(f"{sort_type.name}: no results returned, stopping.")
      break

    prev_count = len(all_reviews)

    for review in result:
      if len(all_reviews) >= target_count:
        break

      review_id = review.get('reviewId')
      if review_id and review_id not in review_ids:
        review_ids.add(review_id)
        all_reviews.append(review)

    added = len(all_reviews) - prev_count
    new_in_window += added
    print(f"{sort_type.name} batch {batch_idx}: +{added} new, total={len(all_reviews)}")

    if len(all_reviews) >= target_count:
      break

    if next_token is None:
      print(f"{sort_type.name}: no continuation token, source exhausted.")
      break

    token_key = repr(next_token)
    if token_key in seen_tokens:
      print(f"{sort_type.name}: repeated continuation token detected, stopping to avoid loop.")
      break

    seen_tokens.add(token_key)
    continuation_token = next_token

    if batch_idx % window_size == 0:
      if new_in_window < min_new_per_window:
        print(f"{sort_type.name}: only {new_in_window} new reviews in last {window_size} batches, stopping.")
        break
      new_in_window = 0

    time.sleep(1.2)

  if len(all_reviews) >= target_count:
    break

  if time.time() - start_time > max_runtime_seconds:
    break

df = pd.DataFrame(all_reviews[:target_count])
print(f"Total reviews collected: {len(df)}.")
if len(df) < target_count:
  print("Target not fully reached. Try running again later for newer reviews.")

NEWEST batch 1: +200 new, total=200
NEWEST batch 2: +200 new, total=400
NEWEST batch 3: +200 new, total=600
NEWEST batch 4: +200 new, total=800
NEWEST batch 5: +200 new, total=1000
NEWEST batch 6: +200 new, total=1200
NEWEST batch 7: +200 new, total=1400
NEWEST batch 8: +200 new, total=1600
NEWEST batch 9: +200 new, total=1800
NEWEST batch 10: +200 new, total=2000
NEWEST batch 11: +200 new, total=2200
NEWEST batch 12: +200 new, total=2400
NEWEST batch 13: +200 new, total=2600
NEWEST batch 14: +200 new, total=2800
NEWEST batch 15: +200 new, total=3000
NEWEST batch 16: +200 new, total=3200
NEWEST batch 17: +200 new, total=3400
NEWEST batch 18: +200 new, total=3600
NEWEST batch 19: +200 new, total=3800
NEWEST batch 20: +200 new, total=4000
NEWEST batch 21: +200 new, total=4200
NEWEST batch 22: +200 new, total=4400
NEWEST batch 23: +200 new, total=4600
NEWEST batch 24: +200 new, total=4800
NEWEST batch 25: +200 new, total=5000
NEWEST batch 26: +200 new, total=5200
NEWEST batch 27: +200 new

In [8]:
# create label sentiment from ratings
def label_sentiment(score):
  if score >= 4:
    return 'positive'
  elif score == 3:
    return 'neutral'
  else:
    return 'negative'

df['sentiment'] = df['score'].apply(label_sentiment)

print(df['sentiment'].value_counts(normalize=True))

sentiment
negative    0.5377
positive    0.4114
neutral     0.0509
Name: proportion, dtype: float64


In [9]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [10]:
def preprocess_text(text):
  text = text.lower()

  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', '', text)

  text = ' '.join(text.split())

  stop_words = set(stopwords.words('english'))
  lemmatizer = WordNetLemmatizer()

  words = text.split()
  words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

  return ' '.join(words)

df['cleaned_content'] = df['content'].apply(preprocess_text)

In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Binary classification: positive vs negative (neutral excluded for cleaner signal)
df_binary = df[df['sentiment'] != 'neutral'].reset_index(drop=True)

texts = df_binary['cleaned_content'].fillna('').values
labels = (df_binary['sentiment'] == 'positive').astype(int).values  # 1=positive, 0=negative

# Tokenize text into integer sequences
MAX_WORDS = 20000
MAX_LEN   = 150

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Split while tracking original indices for CSV export
indices = np.arange(len(df_binary))
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, labels, indices,
    test_size=0.2, random_state=42, stratify=labels
)

print(f"Training set : {X_train.shape}  — positives: {y_train.sum()}/{len(y_train)}")
print(f"Test set     : {X_test.shape}   — positives: {y_test.sum()}/{len(y_test)}")

Training set : (7592, 150)  — positives: 3291/7592
Test set     : (1899, 150)   — positives: 823/1899


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Conv1D,
    Dense, Dropout, SpatialDropout1D, GlobalMaxPooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

EMBED_DIM  = 128
LSTM_UNITS = 128

# Conv1D + BiLSTM hybrid: CNN captures local n-gram patterns,
# BiLSTM captures long-range context in both directions
model = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    Conv1D(128, 5, activation='relu', padding='same'),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=False,
                       dropout=0.2, recurrent_dropout=0.1)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')  # binary output
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

# Compute class weights to handle class imbalance
class_counts = np.bincount(y_train)
class_weight = {
    0: len(y_train) / (2 * class_counts[0]),
    1: len(y_train) / (2 * class_counts[1])
}

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, min_lr=1e-5, verbose=1)
]

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test),
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

best_train_acc = max(history.history['accuracy'])
best_val_acc   = max(history.history['val_accuracy'])
print(f"\nBest Training Accuracy   : {best_train_acc:.2%}")
print(f"Best Validation Accuracy : {best_val_acc:.2%}")

d:\Projects\VSCode Projects\idcamp-intermediate\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 85s 661ms/step - accuracy: 0.8633 - loss: 0.3227 - val_accuracy: 0.9315 - val_loss: 0.2031 - learning_rate: 0.0010
Epoch 2/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 88s 744ms/step - accuracy: 0.9542 - loss: 0.1626 - val_accuracy: 0.9358 - val_loss: 0.1881 - learning_rate: 0.0010
Epoch 3/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 85s 715ms/step - accuracy: 0.9702 - loss: 0.1085 - val_accuracy: 0.9363 - val_loss: 0.2210 - learning_rate: 0.0010
Epoch 4/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 691ms/step - accuracy: 0.9789 - loss: 0.0726
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
119/119 ━━━━━━━━━━━━━━━━━━━━ 87s 733ms/step - accuracy: 0.9771 - loss: 0.0808 - val_accuracy: 0.9268 - val_loss: 0.2450 - learning_rate: 0.0010
Epoch 5/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 86s 721ms/step - accuracy: 0.9880 - loss: 0.0475 - val_accuracy: 0.9263 - val_loss: 0.2816 - learning_rate: 5.0000e-04
Epoch 6/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 690ms/step - accuracy: 0.

In [13]:
import os
from sklearn.metrics import classification_report, accuracy_score

# --- Evaluation ---
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

label_map = {1: 'positive', 0: 'negative'}
y_test_labels = [label_map[l] for l in y_test]
y_pred_labels = [label_map[l] for l in y_pred]

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.2%}\n")
print(classification_report(y_test_labels, y_pred_labels))

# --- CSV Export for reproducibility ---
os.makedirs('data', exist_ok=True)

export_cols = ['content', 'cleaned_content', 'sentiment']

# train.csv — labelled training rows
train_rows = df_binary.iloc[idx_train][export_cols].copy()
train_rows.to_csv('data/train.csv', index=False)

# test.csv — labelled test rows (ground truth)
test_rows = df_binary.iloc[idx_test][export_cols].copy()
test_rows.to_csv('data/test.csv', index=False)

# predictions.csv — test rows with model predictions alongside actuals
pred_rows = test_rows.copy()
pred_rows = pred_rows.rename(columns={'sentiment': 'actual_sentiment'})
pred_rows['predicted_sentiment'] = y_pred_labels
pred_rows.to_csv('data/predictions.csv', index=False)

print(f"\nExported:")
print(f"  data/train.csv       : {len(train_rows):,} rows")
print(f"  data/test.csv        : {len(test_rows):,} rows")
print(f"  data/predictions.csv : {len(pred_rows):,} rows (actual + predicted)")

60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step
Test Accuracy: 93.63%

              precision    recall  f1-score   support

    negative       0.92      0.97      0.95      1076
    positive       0.96      0.89      0.92       823

    accuracy                           0.94      1899
   macro avg       0.94      0.93      0.93      1899
weighted avg       0.94      0.94      0.94      1899


Exported:
  data/train.csv       : 7,592 rows
  data/test.csv        : 1,899 rows
  data/predictions.csv : 1,899 rows (actual + predicted)
